In [11]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler
from pyspark.sql.functions import col
from pyspark.sql.types import FloatType
import numpy as np
from models.fcm import Dfcm
from utils.validity import * 
import time 

In [2]:
# Initialize Spark session
spark = SparkSession.builder.appName("FCM_PySpark").getOrCreate()
spark

24/08/23 18:22:53 WARN Utils: Your hostname, ubuntu resolves to a loopback address: 127.0.1.1; using 172.20.10.9 instead (on interface wlx9803cf079336)
24/08/23 18:22:53 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
24/08/23 18:22:53 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
# Bước 1: Đọc và chuẩn bị dữ liệu
csv_file_path = "data/csv/602_Dry_Bean.csv"
data = spark.read.csv(csv_file_path, header=True, inferSchema=True)
data = data.drop(data.columns[-1])
for column in data.columns:
    data = data.withColumn(column, col(column).cast(FloatType()))

assembler = VectorAssembler(inputCols=data.columns, outputCol="features")
data_vectorized = assembler.transform(data)

In [4]:
# Bước 2: Chuyển đổi thành RDD và phân vùng
k = 4  # Số lượng phân vùng
rdd = data_vectorized.select("features").rdd.map(lambda row: row[0].toArray())
rdd_partitioned = rdd.repartition(k)

print(f"Number of partitions: {rdd_partitioned.getNumPartitions()}")
print("Partition sizes:")
print(rdd_partitioned.glom().map(len).collect())

Number of partitions: 4
Partition sizes:


[3400, 3410, 3401, 3400]


In [28]:
# Lấy dữ liệu trong từng phân vùng
partitioned_data = rdd_partitioned.glom().collect()

fcm = Dfcm()
# Hiển thị dữ liệu của từng phân vùng
_start_time = time.time()

v = np.random.rand(7, 16)
for step in range(10000):
    v_old = v
    Us = []
    for i, partition in enumerate(partitioned_data):
        data_np = np.array(partition)
        sdistances = norm_distances(data_np, v)
        membership = fcm.update_membership_matrix(sdistances)
        Us.append(membership)

    U_final = np.concatenate(Us)
    v = fcm.update_cluster_centers(np.array(rdd.collect()), U_final)

    if (np.abs(v - v_old)).max(axis=(0, 1)) < 1e-5:
        break
    
metric_nt = {
    'Time': round_float(time.time() - _start_time),
    'PC': partition_coefficient(U_final) ,
}
print(metric_nt)    

{'Time': 7.637, 'PC': 0.143}


In [16]:
# Lấy dữ liệu trong từng phân vùng
partitioned_data = rdd_partitioned.glom().collect()

fcm = Dfcm()
# Hiển thị dữ liệu của từng phân vùng
_start_time = time.time()

U = Dfcm.
for step in range(10000):
    
    Us = []
    for i, partition in enumerate(partitioned_data):
        data_np = np.array(partition)
        sdistances = norm_distances(data_np, v)
        membership = fcm.update_membership_matrix(sdistances)
        Us.append(membership)

    U_final = np.concatenate(Us)
    v = fcm.update_cluster_centers(np.array(rdd.collect()), U_final)

    if (np.abs(v - v_old)).max(axis=(0, 1)) < 1e-5:
        break
    
metric_nt = {
    'Time': round_float(time.time() - _start_time),
    'PC': partition_coefficient(U_final) ,
}
print(metric_nt)    


{'Time': 1.246, 'PC': 0.732}


In [29]:
def to_np_array(iterator):
    return np.array(list(iterator))

def update_fcm(rdd_partitioned, k, max_iter=10000, tol=1e-5):
    fcm = Dfcm()
    
    def compute_distances(data_np, v):
        return norm_distances(data_np, v)
    
    def compute_membership(data_np, v):
        sdistances = compute_distances(data_np, v)
        return fcm.update_membership_matrix(sdistances)
    
    def update_centers(rdd, U_final):
        return fcm.update_cluster_centers(np.array(rdd.collect()), U_final)
    
    # Initial cluster centers
    v = np.random.rand(k, len(rdd_partitioned.first()))
    
    _start_time = time.time()
    
    for step in range(max_iter):
        v_old = v
        
        Us = rdd_partitioned.mapPartitions(lambda partition: [compute_membership(to_np_array(partition), v)]).collect()
        U_final = np.concatenate(Us)
        v = update_centers(rdd_partitioned, U_final)
        
        if np.max(np.abs(v - v_old)) < tol:
            break
    
    metric_nt = {
        'Time': round(time.time() - _start_time, 2),
        'PC': partition_coefficient(U_final)
    }
    
    return metric_nt

metrics = update_fcm(rdd_partitioned, k)
print(metrics)

spark.stop()

{'Time': 19.94, 'PC': 0.79}
